In [1]:
from generate_utils import load_GraphModel, load_BiLSTMModel, load_TokenBiLSTMModel, load_LoRASEModel, load_AdapterModel
import torch
import numpy as np
import pickle
from GridMLM_tokenizers import CSGridMLMTokenizer
import os
from eval_utils import get_vecser_for_file, vecser_similarity_matrix
from dotenv import load_dotenv
from tqdm import tqdm
from eval_utils import vecser_similarity_evidence_for_files

from langchain_ollama import ChatOllama
from langchain.tools import tool

# Load environment variables from .env file
load_dotenv()

/home/maximos/miniconda3/envs/torch/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
# Initialize the ChatOllama model with the specified model name
model_name = 'qwen2.5-coder:7b'

# and initialize the ChatOllama instance
chat_model = ChatOllama(
    model=model_name,
    validate_model_on_init=True,
    temperature=0.7
)

In [3]:
tokenizer = CSGridMLMTokenizer(
    fixed_length=80,
    quantization='4th',
    intertwine_bar_info=True,
    trim_start=False,
    use_pc_roll=True,
    use_full_range_melody=False
)

def absoluteFilePaths(directory):
    file_names = []
    file_paths = []
    for dirpath,_,filenames in os.walk(directory):
        for f in filenames:
            if f.endswith( ('.mid', '.midi', '.mxl', '.xml', '.musicxml') ):
                file_names.append(f)
                file_paths.append(os.path.abspath(os.path.join(dirpath, f)))
    return file_names, file_paths
# end absoluteFilePaths

hook_file_names, hook_file_paths = absoluteFilePaths(os.getenv('VAL_HOOK'))
gjt_file_names, gjt_file_paths = absoluteFilePaths(os.getenv('VAL_GJT'))

device_name = 'cuda:2'
device = torch.device(device_name)

guide_arch = 'LoRA'
contra = True

adapter_model_path = f'saved_models/{guide_arch}/adapter/adapter_model_' + contra*'contra_' + 'jnhw.pt'
graph_adapter_model_path = f'saved_models/{guide_arch}/adapter/graph_model_' + contra*'contra_' + 'jnhw.pt'
token_adapter_model_path = f'saved_models/{guide_arch}/adapter/bilstm_model_' + contra*'contra_' + 'jnhw.pt'

token_adapter_model = load_TokenBiLSTMModel(token_adapter_model_path, tokenizer, device)
graph_adapter_model = load_GraphModel(graph_adapter_model_path, device)
adapter_model = load_AdapterModel(adapter_model_path, device)

token_adapter_model.eval()
graph_adapter_model.eval()
adapter_model.eval()

GuidanceAdapter(
  (proj): Linear(in_features=1024, out_features=512, bias=True)
)

In [4]:
f1 = gjt_file_paths[0]
f2 = gjt_file_paths[1]
print(f1)
print(f2)

res = vecser_similarity_evidence_for_files(
    f1,
    f2,
    tokenizer,
    graph_model=graph_adapter_model,
    bilstm_model=None,
    token_model=token_adapter_model,
    adapter_model=adapter_model,
    topk=10
)

print(res)

/media/maindisk/data/mel_harm_CA_all/gjt_CA_test/Mean_To_Me.mxl
/media/maindisk/data/mel_harm_CA_all/gjt_CA_test/My_One_And_Only_Love.mxl
Graph evidence:['A:min7', 'D:7']-['A:min7', 'D:7']similarity value: 1.0['A:min7', 'D:7']-['A:min7', 'D:7']similarity value: 1.0['A:min7', 'D:7']-['A:min7', 'D:7']similarity value: 1.0['A:min7', 'D:7']-['A:min7', 'D:7']similarity value: 1.0['A:min7']-['A:min7']similarity value: 0.9999998807907104['A:min7']-['A:min7']similarity value: 0.9999998807907104['C:maj6']-['A:min7']similarity value: 0.939810037612915['C:maj6']-['A:min7']similarity value: 0.939810037612915['C:maj6']-['A:min7']similarity value: 0.939810037612915['C:maj6']-['A:min7']similarity value: 0.939810037612915
